# Exam 06 — Interview Simulation

**Format:** Big Tech / ML Systems onsite style

| Part | Topic | Questions | Points |
|---|---|---|---|
| I | Coding — Data Structures & Algorithms | 10 | 30 |
| II | ML Engineering — sLM / LLM Systems | 10 | 30 |
| III | System Design | 6 | 24 |
| IV | Debugging & Diagnosis | 6 | 16 |
| **Total** | | **32** | **100** |

> **Rules:** All coding cells run CPU-only, no GPU. Implement the function body directly — no stubs.
---

In [1]:
import sys, types, numpy as np, re, json, time, random, heapq, collections
from unittest.mock import MagicMock
for pkg in ["torch","transformers","peft","bitsandbytes","datasets",
            "sentence_transformers","faiss","chromadb","langchain",
            "langchain_core","langchain_community","langsmith","ragas"]:
    sys.modules.setdefault(pkg, types.ModuleType(pkg))

class FakeEmbed:
    dim = 16
    def embed(self, text):
        v = np.zeros(self.dim)
        for i,w in enumerate(text.lower().split()): v[i%self.dim] += 1
        n = np.linalg.norm(v); return (v/n if n else v).tolist()

def cosine(a, b):
    a,b = np.array(a,float), np.array(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))

class FakeLLM:
    def __init__(self, replies=None):
        self._replies = replies or []; self._idx = 0
    def invoke(self, prompt):
        if self._replies:
            r = self._replies[self._idx % len(self._replies)]; self._idx += 1; return r
        return "mocked response"

class FakeVectorStore:
    def __init__(self): self._docs=[]; self._vecs=[]; self._emb=FakeEmbed()
    def add(self, texts): [self._docs.append(t) or self._vecs.append(self._emb.embed(t)) for t in texts]
    def search(self, q, k=3):
        scores=[cosine(self._emb.embed(q),v) for v in self._vecs]
        top=sorted(range(len(scores)),key=lambda i:scores[i],reverse=True)[:k]
        return [self._docs[i] for i in top]

print("Mock OK")


Mock OK


---
## Part I — Coding: DS&A (10 × 3 pts)

*Implement each function completely. Auto-grader verifies correctness.*

### C01. Top-K frequent words
Given a list of strings `words` and integer `k`, return the `k` most frequent words sorted by frequency (desc), then lexicographically (asc) on ties.

In [2]:
def top_k_frequent_words(words, k):
    from collections import Counter
    cnt = Counter(words)
    return sorted(cnt, key=lambda w: (-cnt[w], w))[:k]


In [3]:
try:
    assert top_k_frequent_words(['the','the','dog','cat','the','dog'],2)==['the','dog']
    assert top_k_frequent_words(['a','b','a','b','c'],1)==['a']
    assert top_k_frequent_words(['a','b'],2)==['a','b']
    print('C01 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C01 PASS


### C02. LRU Cache
Implement `LRUCache(capacity)` with `get(key)` (return -1 if missing) and `put(key, value)` (evict LRU on overflow). O(1) for both operations.

In [4]:
class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self._cache = collections.OrderedDict()
    def get(self, key):
        if key not in self._cache: return -1
        self._cache.move_to_end(key)
        return self._cache[key]
    def put(self, key, value):
        if key in self._cache: self._cache.move_to_end(key)
        self._cache[key] = value
        if len(self._cache) > self.cap:
            self._cache.popitem(last=False)


In [5]:
try:
    c=LRUCache(2)
    c.put(1,1);c.put(2,2);assert c.get(1)==1
    c.put(3,3);assert c.get(2)==-1
    c.put(4,4);assert c.get(1)==-1;assert c.get(3)==3;assert c.get(4)==4
    print('C02 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C02 PASS


### C03. Sliding window max
Given array `nums` and window size `k`, return the maximum for each window position. Use a deque for O(N) time.

In [6]:
def sliding_window_max(nums, k):
    from collections import deque
    dq = deque()  # stores indices, decreasing order of nums[i]
    result = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x: dq.pop()
        dq.append(i)
        if dq[0] < i - k + 1: dq.popleft()
        if i >= k - 1: result.append(nums[dq[0]])
    return result


In [7]:
try:
    assert sliding_window_max([1,3,-1,-3,5,3,6,7],3)==[3,3,5,5,6,7]
    assert sliding_window_max([1],1)==[1]
    print('C03 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C03 PASS


### C04. Merge K sorted lists
Given a list of sorted lists, return a single merged sorted list. Use a min-heap.

In [8]:
def merge_k_sorted(lists):
    h = []
    for i,lst in enumerate(lists):
        if lst: heapq.heappush(h,(lst[0],i,0))
    result = []
    while h:
        val,li,idx = heapq.heappop(h)
        result.append(val)
        if idx+1 < len(lists[li]):
            heapq.heappush(h,(lists[li][idx+1],li,idx+1))
    return result


In [9]:
try:
    assert merge_k_sorted([[1,4,5],[1,3,4],[2,6]])==[1,1,2,3,4,4,5,6]
    assert merge_k_sorted([[]])==[]
    assert merge_k_sorted([[1],[2],[3]])==[1,2,3]
    print('C04 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C04 PASS


### C05. Word break
Given string `s` and list of words `wordDict`, return `True` if `s` can be segmented into words from the dict. DP solution.

In [10]:
def word_break(s, wordDict):
    ws = set(wordDict)
    n = len(s)
    dp = [False]*(n+1); dp[0]=True
    for i in range(1,n+1):
        for j in range(i):
            if dp[j] and s[j:i] in ws:
                dp[i]=True; break
    return dp[n]


In [11]:
try:
    assert word_break('leetcode',['leet','code'])==True
    assert word_break('applepenapple',['apple','pen'])==True
    assert word_break('catsandog',['cats','dog','sand','and','cat'])==False
    print('C05 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C05 PASS


### C06. Trie insert and search
Implement a `Trie` with `insert(word)`, `search(word)` (exact match), and `starts_with(prefix)`.

In [12]:
class Trie:
    def __init__(self): self.root={}
    def insert(self,word):
        n=self.root
        for c in word: n=n.setdefault(c,{})
        n['#']=True
    def search(self,word):
        n=self.root
        for c in word:
            if c not in n: return False
            n=n[c]
        return '#' in n
    def starts_with(self,prefix):
        n=self.root
        for c in prefix:
            if c not in n: return False
            n=n[c]
        return True


In [13]:
try:
    t=Trie()
    t.insert('apple')
    assert t.search('apple')==True
    assert t.search('app')==False
    assert t.starts_with('app')==True
    assert t.starts_with('xyz')==False
    print('C06 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C06 PASS


### C07. Course schedule (cycle detection)
Given `numCourses` and `prerequisites` as `(a,b)` pairs meaning 'a requires b', return `True` if all courses can be finished (no cycle).

In [14]:
def can_finish(numCourses, prerequisites):
    from collections import deque
    in_deg=[0]*numCourses; adj=[[] for _ in range(numCourses)]
    for a,b in prerequisites: adj[b].append(a); in_deg[a]+=1
    q=deque(i for i in range(numCourses) if in_deg[i]==0)
    done=0
    while q:
        u=q.popleft(); done+=1
        for v in adj[u]:
            in_deg[v]-=1
            if in_deg[v]==0: q.append(v)
    return done==numCourses


In [15]:
try:
    assert can_finish(2,[[1,0]])==True
    assert can_finish(2,[[1,0],[0,1]])==False
    assert can_finish(3,[[0,1],[0,2],[1,2]])==True
    print('C07 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C07 PASS


### C08. Longest common subsequence
Return the length of the LCS of strings `s1` and `s2`. Classic DP.

In [16]:
def lcs(s1, s2):
    m,n=len(s1),len(s2)
    dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            if s1[i-1]==s2[j-1]: dp[i][j]=dp[i-1][j-1]+1
            else: dp[i][j]=max(dp[i-1][j],dp[i][j-1])
    return dp[m][n]


In [17]:
try:
    assert lcs('abcde','ace')==3
    assert lcs('abc','abc')==3
    assert lcs('abc','def')==0
    print('C08 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C08 PASS


### C09. Reservoir sampling
Implement `reservoir_sample(stream, k)` where `stream` is an iterator of unknown length. Return exactly k items chosen uniformly at random. Use Algorithm R.

In [18]:
def reservoir_sample(stream, k, seed=42):
    rng=random.Random(seed)
    reservoir=[]
    for i,item in enumerate(stream):
        if i<k: reservoir.append(item)
        else:
            j=rng.randint(0,i)
            if j<k: reservoir[j]=item
    return reservoir


In [19]:
try:
    data=list(range(1000))
    result=reservoir_sample(iter(data),10)
    assert len(result)==10
    assert all(0<=x<1000 for x in result)
    # Repeat 1000 times: each element should appear ~1% of the time
    counts=collections.Counter()
    for _ in range(1000):
        for x in reservoir_sample(iter(range(10)),3,seed=None): counts[x]+=1
    freqs=[counts[i]/1000 for i in range(10)]
    assert all(0.1<f<0.6 for f in freqs),f'Skewed: {freqs}'
    print('C09 PASS (reservoir sampling uniform)')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C09 PASS (reservoir sampling uniform)


### C10. Cosine similarity matrix
Given matrix `X` of shape (N, D), return the N×N cosine similarity matrix. No sklearn — use NumPy only.

In [20]:
def cosine_sim_matrix(X):
    norms=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
    Xn=X/norms
    return Xn@Xn.T


In [21]:
try:
    X=np.array([[1,0,0],[0,1,0],[1,1,0]],dtype=float)
    S=cosine_sim_matrix(X)
    assert S.shape==(3,3)
    assert abs(S[0,0]-1.0)<1e-6
    assert abs(S[0,1])<1e-6           # orthogonal
    assert abs(S[0,2]-1/np.sqrt(2))<1e-6
    print('C10 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

C10 PASS


---
## Part II — ML Engineering: sLM/LLM Systems (10 × 3 pts)

*Implement the function. Grader verifies output properties.*

### M01. LoRA forward pass
Given `W` (base weight, frozen), `A` and `B` (LoRA matrices), `alpha`, `r`, and input `x`, compute the LoRA-augmented output. No torch — use NumPy.

In [22]:
def lora_forward(W, A, B, alpha, r, x):
    # out = W@x + (alpha/r) * B@A@x
    return W@x + (alpha/r) * B@(A@x)


In [23]:
try:
    np.random.seed(0)
    d_in,d_out,rank=8,8,2
    W=np.random.randn(d_out,d_in)
    A=np.random.randn(rank,d_in)
    B=np.random.randn(d_out,rank)
    x=np.random.randn(d_in)
    out=lora_forward(W,A,B,alpha=16,r=rank,x=x)
    expected=W@x+(16/2)*B@A@x
    assert np.allclose(out,expected)
    print('M01 PASS, ||out||=',np.linalg.norm(out).round(4))
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M01 PASS, ||out||=

 107.7133


### M02. SFT cross-entropy with mask
Compute mean cross-entropy loss over positions where `mask[i] == 1`. `logits` is (T, V), `labels` is (T,) with values in [0,V-1], `mask` is (T,) binary.

In [24]:
def sft_loss(logits, labels, mask):
    T,V=logits.shape
    # log-softmax per timestep
    mx=logits.max(axis=1,keepdims=True)
    log_sm=logits-mx-np.log(np.exp(logits-mx).sum(axis=1,keepdims=True))
    nll=-log_sm[np.arange(T),labels]
    valid=mask.sum()
    if valid==0: return 0.0
    return (nll*mask).sum()/valid


In [25]:
try:
    np.random.seed(1)
    T,V=5,10
    logits=np.random.randn(T,V)
    labels=np.random.randint(0,V,T)
    mask=np.array([0,1,1,1,0],dtype=float)
    loss=sft_loss(logits,labels,mask)
    assert 0<loss<20,'unexpected loss range'
    # unmasked positions should not affect result
    loss2=sft_loss(logits+1e6*(1-mask[:,None]),labels,mask)
    assert abs(loss2-loss)>0.1,'first position polluted result'
    print(f'M02 PASS  loss={loss:.4f}')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

  ✗ Incomplete (AssertionError: first position polluted result)


### M03. EWC penalty
Given dict of `fisher` values, `opt_params` (current), and `ref_params` (from previous task), compute the EWC regularisation term: `lambda/2 * sum(F_i * (theta_i - theta_i*)^2)`.

In [26]:
def ewc_penalty(fisher, opt_params, ref_params, lam=0.5):
    penalty=0.0
    for k in fisher:
        diff=opt_params[k]-ref_params[k]
        penalty+=float((fisher[k]*(diff**2)).sum())
    return lam/2*penalty


In [27]:
try:
    fisher={'w':np.array([1.0,2.0,3.0])}
    opt={'w':np.array([1.1,2.2,3.3])}
    ref={'w':np.zeros(3)}
    p=ewc_penalty(fisher,opt,ref,lam=1.0)
    expected=0.5*(1.0*1.1**2+2.0*2.2**2+3.0*3.3**2)
    assert abs(p-expected)<1e-6,p
    print(f'M03 PASS  penalty={p:.4f}')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M03 PASS  penalty=21.7800


### M04. DPO loss (mock)
Given `log_p_win`, `log_p_lose`, `log_p_ref_win`, `log_p_ref_lose`, and `beta`, compute the DPO loss:
`L = -log(sigmoid(beta * ((log_p_win - log_p_ref_win) - (log_p_lose - log_p_ref_lose))))`.

In [28]:
def dpo_loss(log_p_win, log_p_lose, log_p_ref_win, log_p_ref_lose, beta=0.1):
    delta=(log_p_win-log_p_ref_win)-(log_p_lose-log_p_ref_lose)
    return -np.log(1/(1+np.exp(-beta*delta)))


In [29]:
try:
    # perfect policy: win is likely, lose is unlikely vs ref
    loss_good=dpo_loss(-1.0,-5.0,-2.0,-2.0,beta=0.5)
    # bad policy: win is less likely than lose
    loss_bad=dpo_loss(-5.0,-1.0,-2.0,-2.0,beta=0.5)
    assert loss_good<loss_bad,'good policy should have lower loss'
    print(f'M04 PASS  good={loss_good:.4f}  bad={loss_bad:.4f}')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M04 PASS  good=0.1269  bad=2.1269


### M05. Perplexity from log-probs
Given a list of per-token log-probabilities `log_probs` (natural log), compute perplexity: `exp(-mean(log_probs))`.

In [30]:
def perplexity(log_probs):
    return float(np.exp(-np.mean(log_probs)))


In [31]:
try:
    # Perfect prediction: log_prob=-0 would give ppl=1; log_prob=-log(1/V) gives ppl=V
    V=100
    uniform_lp=[-np.log(V)]*50
    ppl=perplexity(uniform_lp)
    assert abs(ppl-V)<1e-4,ppl
    # Good model: log_prob close to 0
    good=perplexity([-0.05]*50)
    assert good<2,'good model should have low ppl'
    print(f'M05 PASS  uniform_ppl={ppl:.1f}  good_ppl={good:.4f}')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M05 PASS  uniform_ppl=100.0  good_ppl=1.0513


### M06. BWT and FWT metrics
Given an accuracy matrix `acc[i][j]` = accuracy on task j after training on task i (rows=tasks trained so far, cols=tasks evaluated), compute `BWT` (backward transfer) and `FWT` (forward transfer).

In [32]:
def bwt_fwt(acc):
    T=len(acc)
    # BWT: avg(acc[T-1][j] - acc[j][j]) for j in 0..T-2
    bwt=np.mean([acc[T-1][j]-acc[j][j] for j in range(T-1)]) if T>1 else 0.0
    # FWT: avg(acc[j-1][j] - acc_random[j]) for j in 1..T-1 (acc_random=0 baseline)
    fwt=np.mean([acc[j-1][j] for j in range(1,T)]) if T>1 else 0.0
    return float(bwt),float(fwt)


In [33]:
try:
    acc=[[0.9,0.0,0.0],[0.8,0.85,0.0],[0.7,0.75,0.88]]
    bwt,fwt=bwt_fwt(acc)
    # BWT should be negative (forgetting)
    assert bwt<0,f'expected negative BWT, got {bwt}'
    # FWT: avg of acc[0][1], acc[1][2] = (0.0+0.0)/2=0.0 (no forward transfer before training)
    print(f'M06 PASS  BWT={bwt:.4f}  FWT={fwt:.4f}')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M06 PASS  BWT=-0.1500  FWT=0.0000


### M07. Warmup LR schedule
Implement `warmup_cosine_lr(step, warmup_steps, total_steps, base_lr)`. Linear warmup from 0 to `base_lr`, then cosine decay to 0.

In [34]:
def warmup_cosine_lr(step, warmup_steps, total_steps, base_lr):
    import math
    if step < warmup_steps:
        return base_lr * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


In [35]:
try:
    import math
    lr0=warmup_cosine_lr(0,100,1000,1e-3)
    lr50=warmup_cosine_lr(50,100,1000,1e-3)
    lr100=warmup_cosine_lr(100,100,1000,1e-3)
    lr1000=warmup_cosine_lr(1000,100,1000,1e-3)
    assert lr0==0.0
    assert abs(lr50-5e-4)<1e-8
    assert abs(lr100-1e-3)<1e-8
    assert lr1000<1e-6
    print(f'M07 PASS  lr0={lr0}  lr50={lr50:.4f}  lr_peak={lr100:.4f}  lr_end={lr1000:.2e}')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M07 PASS  lr0=0.0  lr50=0.0005  lr_peak=0.0010  lr_end=0.00e+00


### M08. Semantic cache lookup
Implement `CachedInference(llm, embed_fn, threshold=0.85)` with `query(text)` that returns a cached response if cosine similarity ≥ threshold, otherwise calls llm and caches the result.

In [36]:
class CachedInference:
    def __init__(self, llm, embed_fn, threshold=0.85):
        self.llm=llm; self.embed_fn=embed_fn; self.threshold=threshold
        self._keys=[]; self._vecs=[]; self._vals=[]
    def query(self, text):
        qv=np.array(self.embed_fn(text))
        for i,v in enumerate(self._vecs):
            if cosine(qv,v)>=self.threshold:
                return self._vals[i]
        resp=self.llm.invoke(text)
        self._keys.append(text); self._vecs.append(qv.tolist()); self._vals.append(resp)
        return resp


In [37]:
try:
    calls=[]
    class CountLLM:
        def invoke(self,p): calls.append(p); return 'answer'
    emb=FakeEmbed()
    ci=CachedInference(CountLLM(),emb.embed,threshold=0.7)
    r1=ci.query('What is LoRA?')
    r2=ci.query('What is LoRA?')  # identical → cache hit
    assert len(calls)==1,'expected 1 LLM call, got '+str(len(calls))
    r3=ci.query('Explain quantum computing')  # very different → cache miss
    assert len(calls)==2
    print('M08 PASS  calls=',len(calls))
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

  ✗ Incomplete (AssertionError: )


### M09. Fisher information diagonal (mock)
Given a list of `gradients` (each a numpy array representing per-sample gradients for a single parameter), compute the diagonal Fisher estimate: mean of squared gradients.

In [38]:
def fisher_diagonal(gradients):
    # gradients: list of (param_name, grad_array) tuples
    fisher={}
    for name,g in gradients:
        fisher[name]=fisher.get(name,0)+g**2
    n=len(set(name for name,_ in gradients))
    # actually count per-param samples
    counts={}
    for name,_ in gradients: counts[name]=counts.get(name,0)+1
    return {k:v/counts[k] for k,v in fisher.items()}


In [39]:
try:
    grads=[('w1',np.array([1.0,2.0])),('w1',np.array([3.0,4.0])),('w2',np.array([0.5]))]
    F=fisher_diagonal(grads)
    assert abs(F['w1'][0]-(1**2+3**2)/2)<1e-9
    assert abs(F['w1'][1]-(2**2+4**2)/2)<1e-9
    assert abs(F['w2'][0]-0.25)<1e-9
    print('M09 PASS:',{k:v.round(3).tolist() for k,v in F.items()})
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M09 PASS: {'w1': [5.0, 10.0], 'w2': [0.25]}


### M10. ROUGE-L score
Implement `rouge_l(reference, hypothesis)` — compute F1 of the longest common subsequence (character-level tokens = words) between reference and hypothesis.

In [40]:
def rouge_l(reference, hypothesis):
    ref=reference.lower().split(); hyp=hypothesis.lower().split()
    m,n=len(ref),len(hyp)
    if m==0 or n==0: return 0.0
    # LCS length
    dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j]=dp[i-1][j-1]+1 if ref[i-1]==hyp[j-1] else max(dp[i-1][j],dp[i][j-1])
    lcs=dp[m][n]
    p=lcs/n; r=lcs/m
    return 2*p*r/(p+r) if p+r>0 else 0.0


In [41]:
try:
    r1=rouge_l('the cat sat on the mat','the cat sat on the mat')
    assert abs(r1-1.0)<1e-6,'perfect match'
    r2=rouge_l('the cat sat on the mat','the dog ran in the park')
    assert 0<r2<0.5,'partial match'
    r3=rouge_l('hello world','foo bar baz')
    assert r3==0.0,'no overlap'
    print(f'M10 PASS  perfect={r1:.2f}  partial={r2:.4f}  none={r3:.2f}')
except (AssertionError,TypeError,AttributeError,NotImplementedError,NameError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

M10 PASS  perfect=1.00  partial=0.3333  none=0.00


---
## Part III — System Design (6 × 4 pts)

*Write your answer as markdown. Rubric reveal at the end.*

### SD01. Design a RAG API at scale (Google/Meta)

Design a production RAG API that serves 10,000 QPS with p99 latency < 200ms. The index has 100 million documents. Describe: ingestion pipeline, embedding service, vector store choice, API layer, caching strategy, and failure modes.

*Your answer here.*

### SD02. Design offline eval for fine-tuned sLM (Anthropic/HuggingFace)

You fine-tuned SmolLM2-360M-Instruct on 50K customer support examples with LoRA. Design a complete offline evaluation framework before deploying to production. What metrics, datasets, and pipeline would you build?

*Your answer here.*

### SD03. Explain the training infrastructure for a 7B model with LoRA on 4×A100 (Meta AI/Databricks)

Describe how you would set up distributed LoRA fine-tuning of a 7B model on 4×A100 80GB GPUs. Cover: memory budget, parallelism strategy, gradient checkpointing, and config.

*Your answer here.*

### SD04. Design a multi-tenant LLM serving system (Amazon/OpenAI)

Design a system that serves 1,000 tenants (companies), each with their own fine-tuned LoRA adapter on a shared base model. Tenants have SLA tiers (gold: p99 < 100ms, silver: p99 < 500ms). How do you architect this?

*Your answer here.*

### SD05. Design a production semantic cache for a Q&A chatbot (Stripe/Dropbox)

Your Q&A chatbot spends 80% of its cost on LLM inference, but you notice 35% of incoming queries are paraphrases of previously answered questions. Design a semantic cache that reduces cost while maintaining answer quality.

*Your answer here.*

### SD06. Design the sLM curriculum's continual learning pipeline (research system design)

Design the full system for Course 2's continual learning scenario: 5 tasks arrive sequentially (Task 0=instruction following, T1=QA, T2=code, T3=summarisation, T4=chat). The model must not forget previous tasks. Describe storage, training loop, eval protocol, and adapter strategy.

*Your answer here.*

### Part III Rubric

In [42]:
R={"SD01": ["Ingestion: async pipeline (Kafka → embedding workers → FAISS/Milvus write shards). Batch embedding with sentence-transformers on GPU fleet. Chunk documents to 256 tokens with 10% overlap.", "Vector store: Milvus or Weaviate clustered across 10 shards, each holding 10M vectors. IVF-PQ index for 10x compression. Hot tier: top-1M most-queried vectors in HNSW on RAM.", "Embedding service: stateless replicas behind a load balancer. Cache embeddings by (text_hash, model_version) in Redis with 1h TTL. At 10K QPS, ~70% cache hit reduces GPU calls to ~3K QPS.", "API layer: FastAPI with async handlers. Semantic cache at the full RAG response level (cosine threshold 0.85). LRU eviction, 100K entry cache. Cache hit rate: ~40% for typical user bases.", "Failure modes: (1) stale index — documents added after last rebuild won't be retrieved; mitigate with incremental index updates; (2) embedding drift — model upgrade invalidates cached embeddings; mitigate with versioned cache keys; (3) shard failure — add replica shard with synchronous writes for consistency.", "Monitoring: p50/p99 latency per shard, cache hit rate, embedding throughput, document staleness (time since last index rebuild)."], "SD02": ["Golden set: 500 human-labelled query-answer pairs spanning: FAQ (easy), multi-step troubleshooting (medium), edge cases (hard). Split 80/20 train/eval.", "Metrics: (1) ROUGE-L vs reference answers; (2) BERTScore semantic similarity; (3) task accuracy on binary outcomes (resolved/not); (4) faithfulness if RAG is involved (RAGAS); (5) safety classifier score (no harmful output).", "Regression suite: 200 queries from the pre-fine-tune model where it already performed well. Fine-tuning must not regress below baseline on these.", "Pipeline: generate → score → compare to baseline → accept if all metrics > expected_band. Fail CI if regression detected. Log all outputs + scores to JSONL for audit.", "Continual learning concern: if using EWC, measure BWT on Task 0 (general instruction following) to ensure catastrophic forgetting didn't occur.", "Shadow deployment: route 1% of production traffic to fine-tuned model, log human feedback, compare satisfaction rate to baseline before full rollout."], "SD03": ["Memory: 7B float16 = ~14GB. LoRA adds ~50MB. Optimizer states (AdamW): 14GB → with gradient checkpointing, activation memory ~4GB per GPU. Total: ~18GB/GPU — fits in 80GB.", "Parallelism: for LoRA on 4 GPUs, data parallelism (DDP or FSDP) is sufficient. Each GPU holds the full model. FSDP with `ShardingStrategy.FULL_SHARD` reduces to ~4GB/GPU for parameters, freeing memory for larger batch size.", "Gradient checkpointing: `model.gradient_checkpointing_enable()` trades compute for memory — recomputes activations on backward instead of storing them. Increases training time ~30% but reduces activation memory from O(layers) to O(1).", "QLoRA variant: load model in NF4 (bitsandbytes) → 7B in ~3.5GB. LoRA adapters in bf16. Total ~5GB/GPU → can run on consumer A10G/3090 GPUs.", "Config: `r=16, alpha=32, target_modules=['q_proj','v_proj'], lora_dropout=0.05`. lr=2e-4, warmup_ratio=0.03, batch_size=4, grad_accum=4 → effective batch 16.", "Monitoring: log GPU memory per rank, gradient norm, validation loss every 100 steps. Early stop if validation loss plateaus for 3 consecutive evals."], "SD04": ["Base model: load once on GPU, shared across all tenants. LoRA adapters are small (~50MB each for 7B with r=16). Load adapters dynamically per request using `set_adapter(tenant_id)`.", "Adapter storage: adapters stored in object storage (S3/GCS). LRU cache of 50 hot adapters in GPU VRAM (~2.5GB). Cold adapters loaded from S3 on first request (~500ms latency). Warm-up period pre-loads top-50 adapters at startup.", "Request routing: gold SLA tenants routed to dedicated GPU instances with adapters pre-loaded. Silver tenants routed to shared pool with dynamic loading. SLA enforced via priority queue in the scheduler.", "Batching: requests from tenants sharing the same adapter can be batched together. Continuous batching (vLLM style) processes multiple tenants' tokens in flight simultaneously — but only possible if all share the same active adapter.", "Multi-adapter serving (LoRAX/S-LoRA): purpose-built systems that can hold 100s of adapters in memory by decomposing the adapter layers and batching across adapters. Enables true multi-tenant efficiency.", "Failure mode: cold start latency. Mitigate with predictive pre-loading based on historical access patterns. Also: adapter version mismatch — version tag in cache key ensures correct adapter loaded after updates."], "SD05": ["Cache architecture: Redis with vector extension (RedisVL or pgvector). Store (embedding_vector, query_text, answer, timestamp, model_version) per entry.", "Lookup flow: embed incoming query → ANN search (k=1) in cache → if cosine similarity ≥ threshold (0.85-0.90), return cached answer → else call LLM → store result.", "Threshold tuning: too low (0.70) → false cache hits (semantically different queries get same answer) → users complain. Too high (0.95) → low hit rate, no cost savings. A/B test: measure hit rate and user satisfaction rating vs threshold.", "Cache invalidation: TTL of 24h for factual queries. For time-sensitive queries (contains 'today', 'current', 'latest'), set TTL=1h or skip cache entirely. On model upgrade, flush all entries (or version-key them).", "Cost model: with 35% hit rate, 50K daily queries, 1000 tokens/query, at $0.002/1K tokens → savings = 0.35×50K×1K/1000×0.002 = $35/day. Cache infrastructure cost ~$5/day. Net savings $30/day.", "Quality assurance: sample 1% of cache hits, route to LLM as well, compare outputs — flag if ROUGE-L < 0.7 for human review. Helps detect threshold misconfiguration."], "SD06": ["Storage: per-task replay buffer (reservoir sampling, 1000 examples each). Fisher matrices stored as numpy arrays alongside model checkpoints. Total: 5×1000 examples + 5 Fisher dicts.", "Training loop: for each new task: (1) load current model; (2) compute Fisher on task data; (3) train with EWC penalty (lambda=0.5); (4) update reservoir buffer; (5) evaluate all previous tasks; (6) log BWT/FWT.", "LoRA-per-task: alternative — freeze base model, train a separate LoRA adapter per task. Inference: load base + task adapter (selected by task classifier or user-specified). Zero forgetting by construction. Downside: inference cost grows linearly with tasks.", "Eval protocol: after each task, run eval on all T+1 tasks. Build accuracy matrix acc[i][j]. Compute BWT (should be > -0.05), FWT (ideally > 0.0). Fail if BWT < -0.10 — catastrophic forgetting detected.", "Task boundary detection: if no explicit task labels, detect boundary when validation loss spikes > 20% above rolling average. Trigger new LoRA adapter or EWC Fisher update.", "Production variant: run eval on a held-out golden set for each task every epoch. If any task drops below baseline × 0.95, pause training and alert. EWC lambda is tuned per task pair based on task similarity (measured by Fisher overlap)."]}
for k,pts in R.items():
    print(f'\n── {k} ──')
    for i,p in enumerate(pts,1): print(f'  {i}. {p}')


── SD01 ──


  1. Ingestion: async pipeline (Kafka → embedding workers → FAISS/Milvus write shards). Batch embedding with sentence-transformers on GPU fleet. Chunk documents to 256 tokens with 10% overlap.
  2. Vector store: Milvus or Weaviate clustered across 10 shards, each holding 10M vectors. IVF-PQ index for 10x compression. Hot tier: top-1M most-queried vectors in HNSW on RAM.
  3. Embedding service: stateless replicas behind a load balancer. Cache embeddings by (text_hash, model_version) in Redis with 1h TTL. At 10K QPS, ~70% cache hit reduces GPU calls to ~3K QPS.
  4. API layer: FastAPI with async handlers. Semantic cache at the full RAG response level (cosine threshold 0.85). LRU eviction, 100K entry cache. Cache hit rate: ~40% for typical user bases.
  5. Failure modes: (1) stale index — documents added after last rebuild won't be retrieved; mitigate with incremental index updates; (2) embedding drift — model upgrade invalidates cached embeddings; mitigate with versioned cache keys; (3)

---
## Part IV — Debugging & Diagnosis (6 × ~2.7 pts)

*Identify the root cause and fix. Explain in prose.*

### D01. RuntimeError: CUDA error: device-side assert triggered

You run a fine-tuning script and see:
```
RuntimeError: CUDA error: device-side assert triggered
```
The traceback points to the cross-entropy loss. What is the most likely cause and how do you diagnose it?

*Your answer here.*

### D02. LoRA training: loss stays at log(V) after 1000 steps

Your LoRA training starts at loss ~4.6 (≈ log(100) for vocab=100) and doesn't decrease after 1000 steps. What went wrong?

*Your answer here.*

### D03. FastAPI lifespan: model loaded on every request

Your FastAPI service has a `/generate` endpoint. Users report the first response takes 8 seconds, but so does every subsequent response. The model should be loaded once. What is wrong?

*Your answer here.*

### D04. ChromaDB: duplicate documents on each run

Every time you restart the RAG service, the document count in ChromaDB doubles. After 5 restarts you have 5× the documents. What is the bug?

*Your answer here.*

### D05. LangGraph: final answer never returned (stuck in tool loop)

Your LangGraph agent always returns `'max steps reached'` and never returns a `Final Answer`. The LLM is generating `Final Answer: ...` in its output but it's being ignored. Diagnose.

*Your answer here.*

### D06. QLoRA: CUDA out of memory on A100 40GB with 7B model

You are running QLoRA on SmolLM2 (135M, much smaller than 7B) but you still get OOM after 2 batches. The model is loaded in NF4. What went wrong?

*Your answer here.*

### Part IV Rubric

In [43]:
R={"D01": ["Root cause: a label value in the batch is out of the valid range [0, vocab_size-1]. This typically happens when (1) the tokenizer produces an unknown token that maps to -1 and it wasn't replaced with -100 in the labels tensor, or (2) the labels tensor was constructed by shifting input_ids but a padding token (e.g. pad_token_id=50256) slipped through.", "Diagnose: add CUDA_LAUNCH_BLOCKING=1 to get a Python-level traceback instead of a cryptic CUDA error. Run on CPU first to see the actual exception.", "Check: print(labels.min(), labels.max()) — if min() is -1 or max() >= vocab_size, that is the bug.", "Fix: in the data collator, set labels[labels == tokenizer.pad_token_id] = -100 before passing to the model. Or use the DataCollatorForSeq2Seq which handles this automatically."], "D02": ["Most likely: LoRA matrices A and B are not being trained — they are accidentally frozen or not registered as parameters.", "Check 1: `print([n for n,p in model.named_parameters() if p.requires_grad])` — only LoRA parameters should appear. If the list is empty, nothing is being trained.", "Check 2: `get_peft_model(model, config)` must be called AFTER `prepare_model_for_kbit_training(model)` for QLoRA. Calling in the wrong order may freeze all params.", "Check 3: learning rate too low (1e-8 vs typical 2e-4 for LoRA). Loss would decrease very slowly.", "Check 4: in `LoraConfig`, `target_modules` lists modules that don't exist in the model. Use `print(model)` to verify module names — for SmolLM2, it is `q_proj`, `v_proj`, not `query`, `value`."], "D03": ["Root cause: the model is being loaded inside the endpoint handler function, not in the lifespan context manager. Every request triggers a new `AutoModelForCausalLM.from_pretrained()` call.", "Fix: use `@asynccontextmanager` lifespan, load the model once, and store it in `app.state.model`. The endpoint accesses it via `request.app.state.model`.", "Anti-pattern: a global variable loaded at import time avoids this but prevents proper cleanup and makes testing harder.", "Diagnosis: add a log line at the start of `from_pretrained` and watch the logs — if it prints on every request, the model is being reloaded."], "D04": ["Root cause: the code unconditionally calls `collection.add(documents=..., ids=...)` on startup without checking if the documents already exist. ChromaDB raises on duplicate IDs (if you provide the same IDs) but if IDs are generated from UUID, they are always unique — so duplicates are inserted.", "Fix 1: use deterministic IDs based on content hash (sha256 of text). Then use `collection.upsert()` instead of `add()` — upsert overwrites if ID already exists.", "Fix 2: check document count on startup. If `collection.count() > 0`, skip ingestion (only works if the corpus never changes).", "Fix 3: use a `last_ingested` timestamp stored in a metadata file. Only ingest documents modified after that timestamp."], "D05": ["Root cause: the regex parser for `Final Answer` is case-sensitive or has a trailing space/punctuation issue. The LLM outputs `Final Answer: ...` but the parser uses a pattern that doesn't match.", "Diagnose: print the raw LLM output before parsing. Check if there is extra whitespace (`Final Answer :`, note the space before colon) or if the LLM wraps it in backticks.", "Fix: make the regex more permissive: `r'Final\\s+Answer\\s*:\\s*(.+?)(?:\\n|$)'` with `re.I` and `re.S` flags. Strip the captured group.", "Check 2: the conditional edge in LangGraph routes based on `state['done']` but `done` is only set if the exact string `'Final Answer'` appears in `state['output']`. If the parser returns `None` for `final_answer`, `done` is never set to `True`."], "D06": ["Check 1: `torch.cuda.memory_allocated()` after model load. If it's 2–3× expected, the model may have been loaded in float32 (16× memory vs NF4) despite `load_in_4bit=True`. Verify `BitsAndBytesConfig` is passed to `from_pretrained`.", "Check 2: gradient checkpointing not enabled. For training, activations accumulate with batch. Enable with `model.gradient_checkpointing_enable()`.", "Check 3: the dataloader is loading the entire dataset into GPU memory (accident like `dataset.to_device('cuda')` on the whole corpus).", "Check 4: `CUDA_VISIBLE_DEVICES` is pointing to a GPU that another process is using — effective free memory is much less than 40GB. Run `nvidia-smi` and verify the GPU is actually free.", "Fix: set `per_device_train_batch_size=1, gradient_accumulation_steps=8` to achieve effective batch 8 with minimal activation memory. Use `max_seq_length=256` to cap sequence length."]}
for k,pts in R.items():
    print(f'\n── {k} ──')
    for i,p in enumerate(pts,1): print(f'  {i}. {p}')


── D01 ──
  1. Root cause: a label value in the batch is out of the valid range [0, vocab_size-1]. This typically happens when (1) the tokenizer produces an unknown token that maps to -1 and it wasn't replaced with -100 in the labels tensor, or (2) the labels tensor was constructed by shifting input_ids but a padding token (e.g. pad_token_id=50256) slipped through.
  2. Diagnose: add CUDA_LAUNCH_BLOCKING=1 to get a Python-level traceback instead of a cryptic CUDA error. Run on CPU first to see the actual exception.
  3. Check: print(labels.min(), labels.max()) — if min() is -1 or max() >= vocab_size, that is the bug.
  4. Fix: in the data collator, set labels[labels == tokenizer.pad_token_id] = -100 before passing to the model. Or use the DataCollatorForSeq2Seq which handles this automatically.

── D02 ──
  1. Most likely: LoRA matrices A and B are not being trained — they are accidentally frozen or not registered as parameters.
  2. Check 1: `print([n for n,p in model.named_parameter